<a href="https://colab.research.google.com/github/saiKelkar/Product_Oriented_Scan-to-CAD_Dimensional_Inspection_System/blob/main/scan_to_cad_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install open3d

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/Scan-to-CAD_System')

folders = [
    "data/nominal",
    "data/synthetic",
    "data/real",
    "src",
    "configs",
    "outputs/reports",
    "outputs/visualizations",
    "benchmarks"
]

for folder in folders:
  (PROJECT_ROOT / folder).mkdir(
      parents=True,
      exist_ok=True
  )

print("Project folders created!")

for folder in folders:
  print(PROJECT_ROOT / folder)

Project folders created!
/content/drive/MyDrive/Scan-to-CAD_System/data/nominal
/content/drive/MyDrive/Scan-to-CAD_System/data/synthetic
/content/drive/MyDrive/Scan-to-CAD_System/data/real
/content/drive/MyDrive/Scan-to-CAD_System/src
/content/drive/MyDrive/Scan-to-CAD_System/configs
/content/drive/MyDrive/Scan-to-CAD_System/outputs/reports
/content/drive/MyDrive/Scan-to-CAD_System/outputs/visualizations
/content/drive/MyDrive/Scan-to-CAD_System/benchmarks


In [4]:
CAD_PATH = (
    PROJECT_ROOT /
    "data/nominal/bracket.stl"
)

SCAN_PATH = (
    PROJECT_ROOT /
    "data/synthetic/raw_scan.ply"
)

GROUND_TRUTH_PATH = (
    PROJECT_ROOT /
    "data/synthetic/ground_truth.json"
)

CONFIG_PATH = (
    PROJECT_ROOT /
    "configs/inspection_config.json"
)

for name, path in [
    ("Nominal CAD", CAD_PATH),
    ("Raw scan", SCAN_PATH),
    ("Ground truth", GROUND_TRUTH_PATH),
    ("Configuration", CONFIG_PATH)
]:
    print(
        f"{name}:",
        "FOUND" if path.exists() else "MISSING"
    )

Nominal CAD: FOUND
Raw scan: FOUND
Ground truth: FOUND
Configuration: FOUND


In [5]:
import open3d as o3d

mesh = o3d.io.read_triangle_mesh(CAD_PATH)
source = o3d.io.read_point_cloud(SCAN_PATH)

In [6]:
%%writefile /content/geometry.py

from pathlib import Path

import numpy as np
import open3d as o3d


def load_cad(path):
    """Load and validate a nominal CAD triangle mesh."""

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            f"CAD file not found: {path}"
        )

    mesh = o3d.io.read_triangle_mesh(str(path))

    if mesh.is_empty() or len(mesh.triangles) == 0:
        raise ValueError("CAD mesh is empty or invalid.")

    vertices = np.asarray(mesh.vertices)

    if not np.all(np.isfinite(vertices)):
        raise ValueError(
            "CAD contains non-finite vertex coordinates."
        )

    mesh.compute_triangle_normals()
    mesh.compute_vertex_normals()

    return mesh


def load_scan(path):
    """Load and validate a 3D point cloud."""

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            f"Scan file not found: {path}"
        )

    scan = o3d.io.read_point_cloud(str(path))

    if scan.is_empty():
        raise ValueError("Scan point cloud is empty.")

    points = np.asarray(scan.points)

    if not np.all(np.isfinite(points)):
        raise ValueError(
            "Scan contains non-finite coordinates."
        )

    return scan


def get_geometry_info(geometry):
    """Return basic geometric information."""

    bbox = geometry.get_axis_aligned_bounding_box()

    info = {
        "dimensions": bbox.get_extent().tolist(),
        "min_bound": bbox.get_min_bound().tolist(),
        "max_bound": bbox.get_max_bound().tolist()
    }

    if isinstance(geometry, o3d.geometry.TriangleMesh):
        info["vertices"] = len(geometry.vertices)
        info["triangles"] = len(geometry.triangles)

    elif isinstance(geometry, o3d.geometry.PointCloud):
        info["points"] = len(geometry.points)

    return info

Overwriting /content/geometry.py


In [7]:
import shutil

GEOMETRY_MODULE = PROJECT_ROOT / "src/geometry.py"

GEOMETRY_MODULE.parent.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy(
    "/content/geometry.py",
    GEOMETRY_MODULE
)

print("Module saved:", GEOMETRY_MODULE)

Module saved: /content/drive/MyDrive/Scan-to-CAD_System/src/geometry.py


In [8]:
import sys

SRC_PATH = str(PROJECT_ROOT / "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from geometry import (
    load_cad,
    load_scan,
    get_geometry_info
)

In [9]:
CAD_PATH = PROJECT_ROOT / "data/nominal/bracket.stl"
SCAN_PATH = PROJECT_ROOT / "data/synthetic/raw_scan.ply"

nominal_mesh = load_cad(CAD_PATH)
raw_scan = load_scan(SCAN_PATH)

print("CAD loaded successfully!")
print("Scan loaded successfully!")

CAD loaded successfully!
Scan loaded successfully!


In [10]:
cad_info = get_geometry_info(nominal_mesh)
scan_info = get_geometry_info(raw_scan)

print("CAD information:")
print(cad_info)

print("\nRaw scan information:")
print(scan_info)

CAD information:
{'dimensions': [15.0, 49.5, 49.5], 'min_bound': [-1.0, -1.0, 0.0], 'max_bound': [14.0, 48.5, 49.5], 'vertices': 2128, 'triangles': 1140}

Raw scan information:
{'dimensions': [43.6898011690686, 58.43016341717748, 59.260623717404386], 'min_bound': [-5.697667182620879, 29.383679092872583, 10.094953994838518], 'max_bound': [37.992133986447726, 87.81384251005007, 69.3555777122429], 'points': 9122}


In [11]:
try:
    load_cad("nonexistent_bracket.stl")

except FileNotFoundError as e:
    print("Expected error caught:")
    print(e)

Expected error caught:
CAD file not found: nonexistent_bracket.stl


In [12]:
from geometry import load_cad, load_scan, get_geometry_info

nominal_mesh = load_cad(CAD_PATH)
raw_scan = load_scan(SCAN_PATH)

print("CAD loaded successfully!")
print("Scan loaded successfully!")

print("\nCAD information:")
print(get_geometry_info(nominal_mesh))

print("\nRaw scan information:")
print(get_geometry_info(raw_scan))

CAD loaded successfully!
Scan loaded successfully!

CAD information:
{'dimensions': [15.0, 49.5, 49.5], 'min_bound': [-1.0, -1.0, 0.0], 'max_bound': [14.0, 48.5, 49.5], 'vertices': 2128, 'triangles': 1140}

Raw scan information:
{'dimensions': [43.6898011690686, 58.43016341717748, 59.260623717404386], 'min_bound': [-5.697667182620879, 29.383679092872583, 10.094953994838518], 'max_bound': [37.992133986447726, 87.81384251005007, 69.3555777122429], 'points': 9122}


In [13]:
registration_code = '''
import copy
import numpy as np
import open3d as o3d


def register_scan(nominal_mesh, raw_scan, config):
    """
    Align raw scan to nominal CAD.

    Returns:
        registered_scan
        diagnostics
    """

    cfg = config["registration"]
    voxel = cfg["voxel_size_mm"]

    # 1. Reference point cloud
    target = nominal_mesh.sample_points_poisson_disk(
        number_of_points=10000
    )

    source = copy.deepcopy(raw_scan)

    # 2. Downsample
    source_down = source.voxel_down_sample(voxel)
    target_down = target.voxel_down_sample(voxel)

    # 3. Estimate normals
    normal_radius = voxel * cfg["normal_radius_multiplier"]

    for cloud in [source_down, target_down]:
        cloud.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(
                radius=normal_radius,
                max_nn=30
            )
        )

    # 4. Compute FPFH
    feature_radius = voxel * cfg["fpfh_radius_multiplier"]

    def compute_fpfh(cloud):
        return o3d.pipelines.registration.compute_fpfh_feature(
            cloud,
            o3d.geometry.KDTreeSearchParamHybrid(
                radius=feature_radius,
                max_nn=100
            )
        )

    source_fpfh = compute_fpfh(source_down)
    target_fpfh = compute_fpfh(target_down)

    # 5. RANSAC coarse registration
    distance = voxel * cfg["ransac_distance_multiplier"]

    ransac = (
        o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
            source_down,
            target_down,
            source_fpfh,
            target_fpfh,
            mutual_filter=True,
            max_correspondence_distance=distance,
            estimation_method=(
                o3d.pipelines.registration.TransformationEstimationPointToPoint(
                    False
                )
            ),
            ransac_n=3,
            checkers=[
                o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(
                    0.9
                ),
                o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(
                    distance
                )
            ],
            criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(
                100000, 0.999
            )
        )
    )

    # 6. Full ICP
    full_icp = o3d.pipelines.registration.registration_icp(
        source,
        target,
        cfg["icp_distance_mm"],
        ransac.transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        o3d.pipelines.registration.ICPConvergenceCriteria(
            max_iteration=100
        )
    )

    # 7. Transform a temporary source for stable-region selection
    source_coarse = copy.deepcopy(source)
    source_coarse.transform(full_icp.transformation)

    source_points = np.asarray(source_coarse.points)
    target_points = np.asarray(target.points)

    y_max = cfg["stable_region_y_max_mm"]

    source_indices = np.where(source_points[:, 1] < y_max)[0]
    target_indices = np.where(target_points[:, 1] < y_max)[0]

    source_stable = source_coarse.select_by_index(
        source_indices.tolist()
    )

    target_stable = target.select_by_index(
        target_indices.tolist()
    )

    # 8. Stable-region ICP
    stable_icp = o3d.pipelines.registration.registration_icp(
        source_stable,
        target_stable,
        cfg["icp_distance_mm"],
        np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        o3d.pipelines.registration.ICPConvergenceCriteria(
            max_iteration=100
        )
    )

    # Compose transformations
    final_transform = (
        stable_icp.transformation @ full_icp.transformation
    )

    # 9. Apply to original scan
    registered_scan = copy.deepcopy(raw_scan)
    registered_scan.transform(final_transform)

    # 10. Evaluate final alignment against the same reference
    final_eval = o3d.pipelines.registration.evaluate_registration(
        raw_scan,
        target,
        cfg["icp_distance_mm"],
        final_transform
    )

    diagnostics = {
        "ransac_fitness": float(ransac.fitness),
        "ransac_rmse_mm": float(ransac.inlier_rmse),
        "full_icp_fitness": float(full_icp.fitness),
        "full_icp_rmse_mm": float(full_icp.inlier_rmse),
        "stable_icp_fitness": float(stable_icp.fitness),
        "stable_icp_rmse_mm": float(stable_icp.inlier_rmse),
        "final_fitness": float(final_eval.fitness),
        "final_rmse_mm": float(final_eval.inlier_rmse),
        "stable_source_points": int(len(source_stable.points)),
        "stable_target_points": int(len(target_stable.points)),
        "transformation": final_transform.tolist()
    }

    return registered_scan, diagnostics
'''

REGISTRATION_PATH = PROJECT_ROOT / "src/registration.py"
REGISTRATION_PATH.parent.mkdir(parents=True, exist_ok=True)
REGISTRATION_PATH.write_text(registration_code)

print("Saved:", REGISTRATION_PATH)

Saved: /content/drive/MyDrive/Scan-to-CAD_System/src/registration.py


In [14]:
import json

CONFIG_PATH = PROJECT_ROOT / "configs/inspection_config.json"

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

print("Configuration loaded!")

Configuration loaded!


In [15]:
# Add missing registration parameters
defaults = {
    "voxel_size_mm": 1.0,
    "normal_radius_multiplier": 2.0,
    "fpfh_radius_multiplier": 5.0,
    "ransac_distance_multiplier": 1.5,
    "icp_distance_mm": 3.0,
    "stable_region_y_max_mm": 25.0
}

config.setdefault("registration", {})

for key, value in defaults.items():
    config["registration"].setdefault(key, value)

# Save the updated configuration
with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

print("Registration configuration updated!")

print(json.dumps(config["registration"], indent=2))

Registration configuration updated!
{
  "voxel_size_mm": 1.0,
  "normal_radius_multiplier": 2.0,
  "ransac_distance_multiplier": 1.5,
  "icp_distance_mm": 3.0,
  "stable_region_y_max_mm": 25.0,
  "fpfh_radius_multiplier": 5.0
}


In [16]:
import sys
import importlib

SRC_PATH = str(PROJECT_ROOT / "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import registration
importlib.reload(registration)

registered_scan, diagnostics = registration.register_scan(
    nominal_mesh,
    raw_scan,
    config
)

print("Registration completed!")

for key, value in diagnostics.items():
    if key != "transformation":
        print(f"{key}: {value}")

[Open3D WARNING] Too few correspondences (159) after mutual filter, fall back to original correspondences.
Registration completed!
ransac_fitness: 0.5644898907738787
ransac_rmse_mm: 0.9094860819783528
full_icp_fitness: 0.9868449901337426
full_icp_rmse_mm: 0.5205683182093946
stable_icp_fitness: 0.99185667752443
stable_icp_rmse_mm: 0.3327711909396138
final_fitness: 0.9867353650515238
final_rmse_mm: 0.6854997968815397
stable_source_points: 6754
stable_target_points: 7462


In [17]:
from scipy.spatial.transform import Rotation
import numpy as np

GROUND_TRUTH_PATH = (
    PROJECT_ROOT / "data/synthetic/ground_truth.json"
)

with open(GROUND_TRUTH_PATH, "r") as f:
    ground_truth = json.load(f)

T_true = np.array(
    ground_truth["scan_pose"]["scan_to_cad_transformation"]
)

T_est = np.array(diagnostics["transformation"])

R_delta = T_est[:3, :3] @ T_true[:3, :3].T

rotation_error = np.degrees(
    Rotation.from_matrix(R_delta).magnitude()
)

translation_error = np.linalg.norm(
    T_est[:3, 3] - T_true[:3, 3]
)

print(f"Rotation error: {rotation_error:.4f} degrees")
print(f"Translation error: {translation_error:.4f} mm")

Rotation error: 0.2465 degrees
Translation error: 0.1396 mm


In [18]:
def check_registration(diagnostics, config):
    min_fitness = config["quality"]["minimum_registration_fitness"]

    if diagnostics["final_fitness"] < min_fitness:
        return "REVIEW"

    if not np.isfinite(diagnostics["final_rmse_mm"]):
        return "REVIEW"

    if diagnostics["stable_source_points"] < 100:
        return "REVIEW"

    return "ACCEPTABLE"


registration_status = check_registration(diagnostics, config)

print("Registration status:", registration_status)

Registration status: ACCEPTABLE


In [19]:
config["registration"].pop(
    "fpfh_radius_multiplieir",
    None
)

with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

print("Configuration cleaned!")

Configuration cleaned!


In [20]:
features_code = '''
import numpy as np
from scipy.optimize import least_squares


def measure_horizontal_hole(
    points,
    approx_y,
    approx_x,
    search_radius=3.5,
    z_min=1.2,
    z_max=3.2,
    nominal_radius=2.5
):
    """
    Measure a Z-axis cylindrical hole.

    Returns a measurement dictionary.
    Returns REVIEW if fitting is unreliable.
    """

    points = np.asarray(points)

    radial = np.sqrt(
        (points[:, 1] - approx_y)**2 +
        (points[:, 0] - approx_x)**2
    )

    mask = (
        (radial < search_radius) &
        (points[:, 2] > z_min) &
        (points[:, 2] < z_max)
    )

    wall_points = points[mask]

    if len(wall_points) < 50:
        return {
            "status": "REVIEW",
            "reason": "Insufficient candidate wall points",
            "num_points": int(len(wall_points))
        }

    xy = wall_points[:, [1, 0]]

    def residuals(params):
        cy, cx, radius = params

        distances = np.sqrt(
            (xy[:, 0] - cy)**2 +
            (xy[:, 1] - cx)**2
        )

        return distances - radius

    result = least_squares(
        residuals,
        x0=[approx_y, approx_x, nominal_radius],
        bounds=(
            [approx_y - 1.0, approx_x - 1.0, 2.0],
            [approx_y + 1.0, approx_x + 1.0, 3.0]
        )
    )

    cy, cx, radius = result.x

    rmse = np.sqrt(np.mean(result.fun**2))

    # Check optimization and geometric fit quality
    if not result.success or rmse > 0.20:
        return {
            "status": "REVIEW",
            "reason": "Poor circle-fitting quality",
            "num_points": int(len(wall_points)),
            "fit_rmse_mm": float(rmse)
        }

    # Reject fits pressed against parameter bounds
    lower = np.array([
        approx_y - 1.0, approx_x - 1.0, 2.0
    ])

    upper = np.array([
        approx_y + 1.0, approx_x + 1.0, 3.0
    ])

    if np.any(result.x - lower < 0.01) or np.any(
        upper - result.x < 0.01
    ):
        return {
            "status": "REVIEW",
            "reason": "Circle fit reached parameter bounds",
            "num_points": int(len(wall_points))
        }

    return {
        "status": "MEASURED",
        "center_y_mm": float(cy),
        "center_x_mm": float(cx),
        "diameter_mm": float(2 * radius),
        "fit_rmse_mm": float(rmse),
        "num_points": int(len(wall_points))
    }
'''

FEATURES_PATH = PROJECT_ROOT / "src/features.py"

FEATURES_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

FEATURES_PATH.write_text(features_code)

print("Saved:", FEATURES_PATH)

Saved: /content/drive/MyDrive/Scan-to-CAD_System/src/features.py


In [21]:
import sys
import importlib

SRC_PATH = str(PROJECT_ROOT / "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import features
importlib.reload(features)

print("features.py imported successfully!")

features.py imported successfully!


In [22]:
import numpy as np

registered_points = np.asarray(
    registered_scan.points
)

hole1_config = config["features"]["hole_1"]
hole2_config = config["features"]["hole_2"]

hole1_result = features.measure_horizontal_hole(
    registered_points,
    approx_y=hole1_config["approx_center_y_mm"],
    approx_x=hole1_config["approx_center_x_mm"]
)

hole2_result = features.measure_horizontal_hole(
    registered_points,
    approx_y=hole2_config["approx_center_y_mm"],
    approx_x=hole2_config["approx_center_x_mm"]
)

print("Hole 1:")
print(hole1_result)

print("\nHole 2:")
print(hole2_result)

Hole 1:
{'status': 'MEASURED', 'center_y_mm': 15.394298243062824, 'center_x_mm': 6.4480937521553345, 'diameter_mm': 4.989159609419369, 'fit_rmse_mm': 0.10186514022764155, 'num_points': 81}

Hole 2:
{'status': 'MEASURED', 'center_y_mm': 34.48734786893122, 'center_x_mm': 6.507246669319964, 'diameter_mm': 5.019282363441539, 'fit_rmse_mm': 0.10722822148052251, 'num_points': 89}


In [23]:
bad_points = registered_points[:10]

bad_result = features.measure_horizontal_hole(
    bad_points,
    approx_y=15.5,
    approx_x=6.5
)

print(bad_result)

{'status': 'REVIEW', 'reason': 'Insufficient candidate wall points', 'num_points': 0}


In [24]:
plane_code = '''

def fit_plane_svd(points):
    """
    Fit a least-squares plane using SVD.
    Returns centroid, unit normal, and signed residuals.
    """

    points = np.asarray(points, dtype=float)

    if points.ndim != 2 or points.shape[1] != 3:
        raise ValueError("Expected an Nx3 point array.")

    if len(points) < 3:
        raise ValueError("At least 3 points are required.")

    centroid = points.mean(axis=0)
    centered = points - centroid

    _, singular_values, vh = np.linalg.svd(
        centered,
        full_matrices=False
    )

    if singular_values[1] < 1e-8:
        raise ValueError("Points do not define a unique plane.")

    normal = vh[-1]
    normal = normal / np.linalg.norm(normal)

    if normal[2] < 0:
        normal = -normal

    residuals = centered @ normal

    return centroid, normal, residuals


def measure_flange(
    nominal_points,
    scan_points,
    top_z=4.5
):
    """
    Guided horizontal-flange measurement.

    Returns REVIEW if insufficient geometry is available.
    """

    nominal_points = np.asarray(nominal_points)
    scan_points = np.asarray(scan_points)

    # Extract nominal top surface
    nominal_mask = (
        (nominal_points[:, 1] > 8.0) &
        (np.abs(nominal_points[:, 2] - top_z) < 0.05)
    )

    nominal_top = nominal_points[nominal_mask]

    if len(nominal_top) < 100:
        return {
            "status": "REVIEW",
            "reason": "Insufficient nominal top-surface points"
        }

    nominal_centroid, nominal_normal, _ = fit_plane_svd(
        nominal_top
    )

    # Estimate scan normals for guided segmentation
    import open3d as o3d

    cloud = o3d.geometry.PointCloud()
    cloud.points = o3d.utility.Vector3dVector(scan_points)

    cloud.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(
            radius=2.0,
            max_nn=40
        )
    )

    normals = np.asarray(cloud.normals)
    normal_z = np.abs(normals[:, 2])

    # Guided top-flange region
    scan_mask = (
        (scan_points[:, 0] > -1.5) &
        (scan_points[:, 0] < 14.5) &
        (scan_points[:, 1] > 8.0) &
        (scan_points[:, 1] < 49.0) &
        (scan_points[:, 2] > 3.5) &
        (scan_points[:, 2] < 9.0) &
        (normal_z > 0.90)
    )

    scan_top = scan_points[scan_mask]

    if len(scan_top) < 100:
        return {
            "status": "REVIEW",
            "reason": "Insufficient scanned flange points",
            "num_points": int(len(scan_top))
        }

    # Deviations from nominal plane
    deviations = (
        scan_top - nominal_centroid
    ) @ nominal_normal

    # Best-fit manufactured plane
    scan_centroid, scan_normal, residuals = fit_plane_svd(
        scan_top
    )

    angle = np.degrees(
        np.arccos(
            np.clip(
                np.dot(nominal_normal, scan_normal),
                -1.0,
                1.0
            )
        )
    )

    return {
        "status": "MEASURED",
        "num_points": int(len(scan_top)),
        "nominal_plane_deviation_p95_mm": float(
            np.percentile(np.abs(deviations), 95)
        ),
        "nominal_plane_deviation_max_mm": float(
            np.max(np.abs(deviations))
        ),
        "plane_fit_rmse_mm": float(
            np.sqrt(np.mean(residuals**2))
        ),
        "plane_residual_range_mm": float(
            np.ptp(residuals)
        ),
        "angular_deviation_deg": float(angle),
        "nominal_normal": nominal_normal.tolist(),
        "measured_normal": scan_normal.tolist()
    }
'''

with open(FEATURES_PATH, "a") as f:
    f.write(plane_code)

print("Plane-fitting functions added to features.py")

Plane-fitting functions added to features.py


In [25]:
import importlib
import features

importlib.reload(features)

print("Updated features.py loaded!")

Updated features.py loaded!


In [26]:
nominal_sample = nominal_mesh.sample_points_poisson_disk(
    number_of_points=30000
)

nominal_points = np.asarray(nominal_sample.points)
registered_points = np.asarray(registered_scan.points)

print("Nominal points:", len(nominal_points))
print("Registered scan points:", len(registered_points))

Nominal points: 30000
Registered scan points: 9122


In [27]:
flange_result = features.measure_flange(
    nominal_points,
    registered_points,
    top_z=4.5
)

print("Flange measurement:")

for key, value in flange_result.items():
    print(f"{key}: {value}")

Flange measurement:
status: MEASURED
num_points: 1190
nominal_plane_deviation_p95_mm: 2.2704246560991463
nominal_plane_deviation_max_mm: 4.377399497427562
plane_fit_rmse_mm: 0.46379088631404586
plane_residual_range_mm: 5.451683137927235
angular_deviation_deg: 3.5789446695019107
nominal_normal: [-3.626357932033787e-05, 1.7749152855810104e-06, 0.9999999993409013]
measured_normal: [0.003476847495847802, -0.06232305354403369, 0.9980499729614937]


In [28]:
bad_flange_result = features.measure_flange(
    nominal_points,
    registered_points[:10],
    top_z=4.5
)

print(bad_flange_result)

{'status': 'REVIEW', 'reason': 'Insufficient scanned flange points', 'num_points': 1}


In [29]:
wrapper_code = '''

def measure_features(nominal_mesh, registered_scan, config):
    """
    Run all configured feature measurements.

    Returns structured measurements.
    """

    import numpy as np

    nominal_sample = nominal_mesh.sample_points_poisson_disk(
        number_of_points=30000
    )

    nominal_points = np.asarray(nominal_sample.points)
    scan_points = np.asarray(registered_scan.points)

    results = {
        "holes": {},
        "hole_spacing": {},
        "horizontal_flange": {}
    }

    for hole_name in ["hole_1", "hole_2"]:

        hole_cfg = config["features"][hole_name]

        approx_y = hole_cfg["approx_center_y_mm"]
        approx_x = hole_cfg["approx_center_x_mm"]

        nominal_result = measure_horizontal_hole(
            nominal_points,
            approx_y,
            approx_x
        )

        measured_result = measure_horizontal_hole(
            scan_points,
            approx_y,
            approx_x
        )

        feature_result = {
            "nominal": nominal_result,
            "measured": measured_result
        }

        if (
            nominal_result["status"] == "MEASURED"
            and measured_result["status"] == "MEASURED"
        ):
            feature_result["diameter_error_mm"] = (
                measured_result["diameter_mm"]
                - nominal_result["diameter_mm"]
            )

            dy = (
                measured_result["center_y_mm"]
                - nominal_result["center_y_mm"]
            )

            dx = (
                measured_result["center_x_mm"]
                - nominal_result["center_x_mm"]
            )

            feature_result["center_offset_2d_mm"] = float(
                np.sqrt(dx**2 + dy**2)
            )

        results["holes"][hole_name] = feature_result

    # Calculate spacing only if both holes were measured
    h1 = results["holes"]["hole_1"]
    h2 = results["holes"]["hole_2"]

    if all(
        h["nominal"]["status"] == "MEASURED"
        and h["measured"]["status"] == "MEASURED"
        for h in [h1, h2]
    ):
        def spacing(a, b):
            return float(np.hypot(
                a["center_y_mm"] - b["center_y_mm"],
                a["center_x_mm"] - b["center_x_mm"]
            ))

        nominal_spacing = spacing(
            h1["nominal"],
            h2["nominal"]
        )

        measured_spacing = spacing(
            h1["measured"],
            h2["measured"]
        )

        results["hole_spacing"] = {
            "status": "MEASURED",
            "nominal_mm": nominal_spacing,
            "measured_mm": measured_spacing,
            "error_mm": measured_spacing - nominal_spacing
        }

    else:
        results["hole_spacing"] = {
            "status": "REVIEW",
            "reason": "One or more holes could not be measured"
        }

    # Measure flange
    results["horizontal_flange"] = measure_flange(
        nominal_points,
        scan_points,
        top_z=4.5
    )

    return results
'''

with open(FEATURES_PATH, "a") as f:
    f.write(wrapper_code)

print("measure_features() added to features.py")

measure_features() added to features.py


In [30]:
import importlib
import features

importlib.reload(features)

<module 'features' from '/content/drive/MyDrive/Scan-to-CAD_System/src/features.py'>

In [31]:
measurements = features.measure_features(
    nominal_mesh,
    registered_scan,
    config
)

import json

print(json.dumps(measurements, indent=2))

{
  "holes": {
    "hole_1": {
      "nominal": {
        "status": "MEASURED",
        "center_y_mm": 15.480225345826186,
        "center_x_mm": 6.474516665555717,
        "diameter_mm": 4.9840607571833155,
        "fit_rmse_mm": 0.0035749698112328946,
        "num_points": 256
      },
      "measured": {
        "status": "MEASURED",
        "center_y_mm": 15.394298243062824,
        "center_x_mm": 6.4480937521553345,
        "diameter_mm": 4.989159609419369,
        "fit_rmse_mm": 0.10186514022764155,
        "num_points": 81
      },
      "diameter_error_mm": 0.005098852236053553,
      "center_offset_2d_mm": 0.08989792735024256
    },
    "hole_2": {
      "nominal": {
        "status": "MEASURED",
        "center_y_mm": 34.543660934288525,
        "center_x_mm": 6.474367216593001,
        "diameter_mm": 4.984089152737876,
        "fit_rmse_mm": 0.0036051504628911765,
        "num_points": 253
      },
      "measured": {
        "status": "MEASURED",
        "center_y_mm": 34.4

In [32]:
for name, hole in measurements["holes"].items():

    measured = hole["measured"]

    print(f"\n{name}")

    print("Status:", measured["status"])

    if measured["status"] == "MEASURED":
        print("Diameter:", measured["diameter_mm"], "mm")
        print("Fitting RMSE:", measured["fit_rmse_mm"], "mm")

print(
    "\nFlange status:",
    measurements["horizontal_flange"]["status"]
)

print(
    "Hole spacing status:",
    measurements["hole_spacing"]["status"]
)


hole_1
Status: MEASURED
Diameter: 4.989159609419369 mm
Fitting RMSE: 0.10186514022764155 mm

hole_2
Status: MEASURED
Diameter: 5.019282363441539 mm
Fitting RMSE: 0.10722822148052251 mm

Flange status: MEASURED
Hole spacing status: MEASURED


In [33]:
inspection_code = '''
import math


def evaluate_tolerance(error, tolerance, uncertainty=None):
    """
    Evaluate a symmetric tolerance.

    uncertainty=None means no validated uncertainty
    was supplied, so only a provisional decision
    can be made.
    """

    if error is None or not math.isfinite(error):
        return "REVIEW"

    if tolerance is None or tolerance < 0:
        return "REVIEW"

    if uncertainty is None:
        return "PASS" if abs(error) <= tolerance else "FAIL"

    if not math.isfinite(uncertainty) or uncertainty < 0:
        return "REVIEW"

    if abs(error) + uncertainty <= tolerance:
        return "PASS"

    if abs(error) - uncertainty > tolerance:
        return "FAIL"

    return "REVIEW"


def evaluate_inspection(measurements, config, registration_status):
    """
    Convert feature measurements into tolerance decisions.

    Returns an inspection report dictionary.
    """

    # Never inspect features if registration is unreliable.
    if registration_status != "ACCEPTABLE":
        return {
            "overall_status": "REVIEW",
            "reason": "Registration quality insufficient",
            "checks": []
        }

    tolerances = config["tolerances"]
    checks = []

    def add_check(name, error, tolerance, unit, quality="MEASURED"):
        if quality != "MEASURED":
            status = "REVIEW"
        else:
            status = evaluate_tolerance(error, tolerance)

        checks.append({
            "feature": name,
            "error": float(error) if error is not None else None,
            "tolerance": float(tolerance),
            "unit": unit,
            "status": status
        })

    # Hole diameter and center position
    for hole_name in ["hole_1", "hole_2"]:
        hole = measurements["holes"][hole_name]

        quality = (
            "MEASURED"
            if hole["nominal"]["status"] == "MEASURED"
            and hole["measured"]["status"] == "MEASURED"
            else "REVIEW"
        )

        add_check(
            f"{hole_name} diameter",
            hole.get("diameter_error_mm"),
            tolerances["hole_diameter_mm"],
            "mm",
            quality
        )

        add_check(
            f"{hole_name} center offset",
            hole.get("center_offset_2d_mm"),
            tolerances["hole_position_mm"],
            "mm",
            quality
        )

    # Hole spacing
    spacing = measurements["hole_spacing"]

    add_check(
        "Hole center spacing",
        spacing.get("error_mm"),
        tolerances["hole_spacing_mm"],
        "mm",
        spacing["status"]
    )

    # Flange
    flange = measurements["horizontal_flange"]

    add_check(
        "Flange nominal-plane deviation P95",
        flange.get("nominal_plane_deviation_p95_mm"),
        tolerances["flange_plane_deviation_mm"],
        "mm",
        flange["status"]
    )

    add_check(
        "Flange angular deviation",
        flange.get("angular_deviation_deg"),
        tolerances["flange_angle_deg"],
        "degrees",
        flange["status"]
    )

    statuses = [item["status"] for item in checks]

    if "FAIL" in statuses:
        overall = "FAIL"
    elif "REVIEW" in statuses:
        overall = "REVIEW"
    else:
        overall = "PASS"

    return {
        "overall_status": overall,
        "decision_basis": "provisional_demo_tolerances",
        "checks": checks,
        "summary": {
            "passed": statuses.count("PASS"),
            "failed": statuses.count("FAIL"),
            "review": statuses.count("REVIEW")
        }
    }
'''

INSPECTION_PATH = PROJECT_ROOT / "src/inspection.py"
INSPECTION_PATH.parent.mkdir(parents=True, exist_ok=True)
INSPECTION_PATH.write_text(inspection_code)

print("Saved:", INSPECTION_PATH)

Saved: /content/drive/MyDrive/Scan-to-CAD_System/src/inspection.py


In [34]:
import importlib
import inspection

importlib.reload(inspection)

results = inspection.evaluate_inspection(
    measurements,
    config,
    registration_status
)

import json
print(json.dumps(results, indent=2))

{
  "overall_status": "FAIL",
  "decision_basis": "provisional_demo_tolerances",
  "checks": [
    {
      "feature": "hole_1 diameter",
      "error": 0.005098852236053553,
      "tolerance": 0.25,
      "unit": "mm",
      "status": "PASS"
    },
    {
      "feature": "hole_1 center offset",
      "error": 0.08989792735024256,
      "tolerance": 0.5,
      "unit": "mm",
      "status": "PASS"
    },
    {
      "feature": "hole_2 diameter",
      "error": 0.03519321070366299,
      "tolerance": 0.25,
      "unit": "mm",
      "status": "PASS"
    },
    {
      "feature": "hole_2 center offset",
      "error": 0.06520904647026411,
      "tolerance": 0.5,
      "unit": "mm",
      "status": "PASS"
    },
    {
      "feature": "Hole center spacing",
      "error": 0.029705668573193122,
      "tolerance": 0.25,
      "unit": "mm",
      "status": "PASS"
    },
    {
      "feature": "Flange nominal-plane deviation P95",
      "error": 2.270776786396989,
      "tolerance": 0.5,
      "

In [35]:
bad_registration_result = inspection.evaluate_inspection(
    measurements,
    config,
    registration_status="REVIEW"
)

print(json.dumps(bad_registration_result, indent=2))

{
  "overall_status": "REVIEW",
  "reason": "Registration quality insufficient",
  "checks": []
}


In [36]:
reporting_code = '''
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import open3d as o3d


def save_report(results, diagnostics, measurements, output_path):
    """Save a structured JSON inspection report."""

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    report = {
        "project": "3D Scan-to-CAD Dimensional Inspection",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "inspection": results,
        "registration": diagnostics,
        "measurements": measurements
    }

    with open(output_path, "w") as f:
        json.dump(report, f, indent=2, allow_nan=False)

    return report


def create_deviation_cloud(
    nominal_mesh,
    registered_scan,
    tolerance_mm=0.5,
    max_display_distance_mm=5.0
):
    """
    Color registered scan points by unsigned
    distance to nominal CAD surface.

    Green: within tolerance
    Red: above tolerance
    Gray: large-distance/unclassified points
    """

    cad_tensor = o3d.t.geometry.TriangleMesh.from_legacy(
        nominal_mesh
    )

    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(cad_tensor)

    points = np.asarray(registered_scan.points)

    distances = scene.compute_distance(
        o3d.core.Tensor(points.astype(np.float32))
    ).numpy()

    colors = np.zeros((len(points), 3))

    good = distances <= tolerance_mm

    failing = (
        (distances > tolerance_mm) &
        (distances <= max_display_distance_mm)
    )

    unclassified = distances > max_display_distance_mm

    colors[good] = [0.0, 0.8, 0.2]
    colors[failing] = [0.95, 0.1, 0.1]
    colors[unclassified] = [0.55, 0.55, 0.55]

    colored_cloud = o3d.geometry.PointCloud()

    colored_cloud.points = o3d.utility.Vector3dVector(
        points.copy()
    )

    colored_cloud.colors = o3d.utility.Vector3dVector(
        colors
    )

    summary = {
        "total_points": int(len(points)),
        "within_tolerance_points": int(good.sum()),
        "above_tolerance_points": int(failing.sum()),
        "unclassified_points": int(unclassified.sum()),
        "median_unsigned_distance_mm": float(
            np.median(distances)
        ),
        "p95_unsigned_distance_mm": float(
            np.percentile(distances, 95)
        )
    }

    return colored_cloud, summary


def save_colored_cloud(cloud, output_path):
    """Save an annotated point cloud as PLY."""

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    success = o3d.io.write_point_cloud(
        str(output_path),
        cloud
    )

    if not success:
        raise IOError(f"Could not save PLY: {output_path}")

    return str(output_path)
'''

REPORTING_PATH = PROJECT_ROOT / "src/reporting.py"

REPORTING_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORTING_PATH.write_text(reporting_code)

print("Saved:", REPORTING_PATH)

Saved: /content/drive/MyDrive/Scan-to-CAD_System/src/reporting.py


In [37]:
import importlib
import reporting

importlib.reload(reporting)

print("reporting.py loaded!")

reporting.py loaded!


In [38]:
colored_cloud, deviation_summary = (
    reporting.create_deviation_cloud(
        nominal_mesh,
        registered_scan,
        tolerance_mm=0.5
    )
)

print("Deviation summary:")

for key, value in deviation_summary.items():
    print(f"{key}: {value}")

Deviation summary:
total_points: 9122
within_tolerance_points: 7277
above_tolerance_points: 1744
unclassified_points: 101
median_unsigned_distance_mm: 0.13117587566375732
p95_unsigned_distance_mm: 1.854271411895752


In [39]:
REPORT_PATH = (
    PROJECT_ROOT /
    "outputs/reports/inspection_report.json"
)

PLY_PATH = (
    PROJECT_ROOT /
    "outputs/visualizations/deviation_cloud.ply"
)

report = reporting.save_report(
    results,
    diagnostics,
    measurements,
    REPORT_PATH
)

saved_ply = reporting.save_colored_cloud(
    colored_cloud,
    PLY_PATH
)

print("JSON saved:", REPORT_PATH)
print("PLY saved:", saved_ply)

JSON saved: /content/drive/MyDrive/Scan-to-CAD_System/outputs/reports/inspection_report.json
PLY saved: /content/drive/MyDrive/Scan-to-CAD_System/outputs/visualizations/deviation_cloud.ply


In [40]:
import json

with open(REPORT_PATH, "r") as f:
    loaded_report = json.load(f)

loaded_cloud = o3d.io.read_point_cloud(
    str(PLY_PATH)
)

print("Overall inspection:",
      loaded_report["inspection"]["overall_status"])

print("Tolerance checks:",
      len(loaded_report["inspection"]["checks"]))

print("Saved cloud points:",
      len(loaded_cloud.points))

print("Cloud has colors:",
      loaded_cloud.has_colors())

Overall inspection: FAIL
Tolerance checks: 7
Saved cloud points: 9122
Cloud has colors: True


In [41]:
import plotly.graph_objects as go
import numpy as np

points = np.asarray(loaded_cloud.points)
colors = np.asarray(loaded_cloud.colors)

rgb_strings = [
    f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"
    for r, g, b in colors
]

fig = go.Figure(go.Scatter3d(
    x=points[:, 0],
    y=points[:, 1],
    z=points[:, 2],
    mode="markers",
    marker=dict(
        size=2,
        color=rgb_strings
    )
))

fig.update_layout(
    title="Automated Scan-to-CAD Deviation Inspection",
    scene=dict(
        xaxis_title="X (mm)",
        yaxis_title="Y (mm)",
        zaxis_title="Z (mm)",
        aspectmode="data"
    ),
    width=900,
    height=700
)

fig.show()

In [42]:
pipeline_code = '''
import json
from pathlib import Path
import numpy as np

from geometry import load_cad, load_scan
from defects import cluster_flange_defects
from registration import register_scan
from features import measure_features
from inspection import evaluate_inspection
from reporting import (
    save_report,
    create_deviation_cloud,
    save_colored_cloud
)


def check_registration(diagnostics, config):
    """Basic registration quality gate."""

    minimum_fitness = config["quality"][
        "minimum_registration_fitness"
    ]

    fitness = diagnostics["final_fitness"]
    rmse = diagnostics["final_rmse_mm"]

    if not np.isfinite(fitness) or fitness < minimum_fitness:
        return "REVIEW"

    if not np.isfinite(rmse):
        return "REVIEW"

    if diagnostics["stable_source_points"] < 100:
        return "REVIEW"

    T = np.asarray(diagnostics["transformation"])

    if T.shape != (4, 4) or not np.all(np.isfinite(T)):
        return "REVIEW"

    return "ACCEPTABLE"


def run_pipeline(cad_path, scan_path, config_path, output_dir):
    """
    Run the complete scan-to-CAD inspection.

    The inspection never reads synthetic ground truth.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Load configuration
    with open(config_path, "r") as f:
        config = json.load(f)

    # 2. Load inputs
    nominal_mesh = load_cad(cad_path)
    raw_scan = load_scan(scan_path)

    # 3. Registration
    registered_scan, diagnostics = register_scan(
        nominal_mesh,
        raw_scan,
        config
    )

    # 4. Registration quality
    registration_status = check_registration(
        diagnostics,
        config
    )

    # Validate expected hole geometry
    if registration_status == "ACCEPTABLE":

        feature_status, feature_diagnostics = (
            check_registration_features(
                registered_scan,
                config
            )
        )

        diagnostics["feature_validation"] = {
            "status": feature_status,
            "holes": feature_diagnostics
        }

        if feature_status != "ACCEPTABLE":
            registration_status = "REVIEW"

    # 5. Stop inspection if registration is unreliable
    if registration_status != "ACCEPTABLE":
        results = {
            "overall_status": "REVIEW",
            "reason": "Registration quality insufficient",
            "checks": [],
            "summary": {
                "passed": 0,
                "failed": 0,
                "review": 1
            }
        }

        measurements = {}

        report = save_report(
            results,
            diagnostics,
            measurements,
            output_dir / "inspection_report.json"
        )

        return report

    # 6. Feature measurements
    measurements = measure_features(
        nominal_mesh,
        registered_scan,
        config
    )

    # 7. Tolerance evaluation
    results = evaluate_inspection(
        measurements,
        config,
        registration_status
    )

    # 8. Detect localized flange defects
    defect_results = cluster_flange_defects(
        nominal_mesh,
        registered_scan,
        config
    )

    # 8. Save JSON
    report = save_report(
        results,
        diagnostics,
        measurements,
        output_dir / "inspection_report.json"
    )

    # 9. Generate and save colored point cloud
    colored_cloud, deviation_summary = create_deviation_cloud(
        nominal_mesh,
        registered_scan,
        tolerance_mm=config["tolerances"][
            "flange_plane_deviation_mm"
        ]
    )

    save_colored_cloud(
        colored_cloud,
        output_dir / "deviation_cloud.ply"
    )

    # 10. Save point-level deviation summary
    report["deviation_summary"] = deviation_summary
    report["defect_detection"] = defect_results

    with open(output_dir / "inspection_report.json", "w") as f:
        json.dump(report, f, indent=2, allow_nan=False)

    return report
'''

PIPELINE_PATH = PROJECT_ROOT / "src/run_pipeline.py"
PIPELINE_PATH.parent.mkdir(parents=True, exist_ok=True)
PIPELINE_PATH.write_text(pipeline_code)

print("Saved:", PIPELINE_PATH)

Saved: /content/drive/MyDrive/Scan-to-CAD_System/src/run_pipeline.py


In [43]:
from pathlib import Path

PIPELINE_PATH = PROJECT_ROOT / "src/run_pipeline.py"

code = PIPELINE_PATH.read_text()

feature_gate_code = '''

def check_registration_features(registered_scan, config):
    """
    Check whether the expected hole features are
    observable after registration.

    Returns ACCEPTABLE or REVIEW.
    """

    import numpy as np
    from features import measure_horizontal_hole

    points = np.asarray(registered_scan.points)

    diagnostics = {}

    for hole_name in ["hole_1", "hole_2"]:

        hole_cfg = config["features"][hole_name]

        result = measure_horizontal_hole(
            points,
            approx_y=hole_cfg["approx_center_y_mm"],
            approx_x=hole_cfg["approx_center_x_mm"]
        )

        diagnostics[hole_name] = result

        if result["status"] != "MEASURED":
            return "REVIEW", diagnostics

    return "ACCEPTABLE", diagnostics
'''

# Insert the function before run_pipeline()
if "def check_registration_features(" not in code:

    marker = "\ndef run_pipeline("

    assert marker in code, "Could not find run_pipeline()"

    code = code.replace(
        marker,
        feature_gate_code + marker
    )

    PIPELINE_PATH.write_text(code)

    print("Feature validation function added successfully!")

else:
    print("Function already exists.")

Feature validation function added successfully!


In [44]:
import importlib
import run_pipeline

importlib.reload(run_pipeline)

print("Updated pipeline loaded!")

print(
    "Feature validation available:",
    hasattr(run_pipeline, "check_registration_features")
)

Updated pipeline loaded!
Feature validation available: True


In [45]:
import importlib
import run_pipeline

importlib.reload(run_pipeline)

OUTPUT_DIR = PROJECT_ROOT / "outputs/end_to_end_test"

report = run_pipeline.run_pipeline(
    cad_path=CAD_PATH,
    scan_path=SCAN_PATH,
    config_path=CONFIG_PATH,
    output_dir=OUTPUT_DIR
)

print("Overall status:", report["inspection"]["overall_status"])

print(
    "Tolerance checks:",
    len(report["inspection"]["checks"])
)

print("Outputs saved to:", OUTPUT_DIR)

[Open3D WARNING] Too few correspondences (159) after mutual filter, fall back to original correspondences.
Overall status: FAIL
Tolerance checks: 7
Outputs saved to: /content/drive/MyDrive/Scan-to-CAD_System/outputs/end_to_end_test


In [46]:
from pathlib import Path
import json
import open3d as o3d

report_path = OUTPUT_DIR / "inspection_report.json"
ply_path = OUTPUT_DIR / "deviation_cloud.ply"

print("JSON exists:", report_path.exists())
print("PLY exists:", ply_path.exists())

with open(report_path, "r") as f:
    saved_report = json.load(f)

print(
    "Overall inspection:",
    saved_report["inspection"]["overall_status"]
)

print(
    "Registration fitness:",
    saved_report["registration"]["final_fitness"]
)

if ply_path.exists():
    cloud = o3d.io.read_point_cloud(str(ply_path))

    print("Saved cloud points:", len(cloud.points))
    print("Has colors:", cloud.has_colors())

JSON exists: True
PLY exists: True
Overall inspection: FAIL
Registration fitness: 0.9868449901337426
Saved cloud points: 9122
Has colors: True


In [47]:
OUTPUT_DIR_2 = PROJECT_ROOT / "outputs/end_to_end_test_2"

report_2 = run_pipeline.run_pipeline(
    cad_path=CAD_PATH,
    scan_path=SCAN_PATH,
    config_path=CONFIG_PATH,
    output_dir=OUTPUT_DIR_2
)

print("Run 1:", report["inspection"]["overall_status"])
print("Run 2:", report_2["inspection"]["overall_status"])

print(
    "Run 1 fitness:",
    report["registration"]["final_fitness"]
)

print(
    "Run 2 fitness:",
    report_2["registration"]["final_fitness"]
)

[Open3D WARNING] Too few correspondences (172) after mutual filter, fall back to original correspondences.
Run 1: FAIL
Run 2: FAIL
Run 1 fitness: 0.9868449901337426
Run 2 fitness: 0.9868449901337426


In [48]:
import numpy as np
from scipy.spatial.transform import Rotation

T1 = np.array(
    report["registration"]["transformation"]
)

T2 = np.array(
    report_2["registration"]["transformation"]
)

# Relative rotation
R_delta = T2[:3, :3] @ T1[:3, :3].T

rotation_difference = np.degrees(
    Rotation.from_matrix(R_delta).magnitude()
)

# Translation-vector difference
translation_difference = np.linalg.norm(
    T2[:3, 3] - T1[:3, 3]
)

print(f"Rotation difference: {rotation_difference:.6f} degrees")
print(f"Translation difference: {translation_difference:.6f} mm")

Rotation difference: 0.070627 degrees
Translation difference: 0.071172 mm


In [49]:
checks_1 = report["inspection"]["checks"]
checks_2 = report_2["inspection"]["checks"]

print(f"{'Feature':38s} {'Run 1':>12s} {'Run 2':>12s} {'Difference':>12s}")

for c1, c2 in zip(checks_1, checks_2):

    assert c1["feature"] == c2["feature"]

    error1 = c1["error"]
    error2 = c2["error"]

    if error1 is None or error2 is None:
        print(f"{c1['feature']:38s} {str(error1):>12s} {str(error2):>12s} {'N/A':>12s}")
        continue

    difference = abs(error2 - error1)

    print(
        f"{c1['feature']:38s} "
        f"{error1:12.5f} "
        f"{error2:12.5f} "
        f"{difference:12.5f}"
    )

Feature                                       Run 1        Run 2   Difference
hole_1 diameter                             0.00589      0.00472      0.00117
hole_1 center offset                        0.09816      0.10343      0.00526
hole_2 diameter                             0.03475      0.02870      0.00606
hole_2 center offset                        0.06901      0.07000      0.00100
Hole center spacing                         0.02934      0.02617      0.00317
Flange nominal-plane deviation P95          2.26700      2.25883      0.00817
Flange angular deviation                    3.55295      3.55331      0.00036


In [50]:
import json

with open(GROUND_TRUTH_PATH, "r") as f:
    ground_truth = json.load(f)

T_true = np.array(
    ground_truth["scan_pose"]["scan_to_cad_transformation"]
)

def calculate_pose_error(T_est, T_true):
    R_delta = T_est[:3, :3] @ T_true[:3, :3].T

    rotation_error = np.degrees(
        Rotation.from_matrix(R_delta).magnitude()
    )

    translation_error = np.linalg.norm(
        T_est[:3, 3] - T_true[:3, 3]
    )

    return rotation_error, translation_error


for name, T_est in [
    ("Run 1", T1),
    ("Run 2", T2)
]:
    rot, trans = calculate_pose_error(T_est, T_true)

    print(f"\n{name}")
    print(f"Rotation error: {rot:.4f} degrees")
    print(f"Translation error: {trans:.4f} mm")


Run 1
Rotation error: 0.2670 degrees
Translation error: 0.1603 mm

Run 2
Rotation error: 0.2605 degrees
Translation error: 0.2072 mm


In [51]:
defects_code = '''
import numpy as np
import open3d as o3d


def cluster_flange_defects(
    nominal_mesh,
    registered_scan,
    config
):
    """
    Identify candidate excessive deviations on the
    horizontal flange using DBSCAN.

    Returns cluster summaries and diagnostics.
    """

    points = np.asarray(registered_scan.points)

    if len(points) == 0:
        return {
            "status": "REVIEW",
            "reason": "Empty registered scan",
            "clusters": []
        }

    # Estimate normals
    cloud = o3d.geometry.PointCloud()
    cloud.points = o3d.utility.Vector3dVector(points)

    cloud.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(
            radius=2.0,
            max_nn=40
        )
    )

    normals = np.asarray(cloud.normals)
    normal_z = np.abs(normals[:, 2])

    # Guided horizontal top-flange segmentation
    mask = (
        (points[:, 0] > -1.5) &
        (points[:, 0] < 14.5) &
        (points[:, 1] > 8.0) &
        (points[:, 1] < 49.0) &
        (points[:, 2] > 3.5) &
        (points[:, 2] < 9.0) &
        (normal_z > 0.90)
    )

    flange_points = points[mask]

    if len(flange_points) < 100:
        return {
            "status": "REVIEW",
            "reason": "Insufficient flange surface points",
            "clusters": []
        }

    # Reference plane from nominal top surface
    nominal_points = np.asarray(
        nominal_mesh.sample_points_poisson_disk(
            number_of_points=30000
        ).points
    )

    nominal_mask = (
        (nominal_points[:, 1] > 8.0) &
        (np.abs(nominal_points[:, 2] - 4.5) < 0.05)
    )

    nominal_top = nominal_points[nominal_mask]

    if len(nominal_top) < 100:
        return {
            "status": "REVIEW",
            "reason": "Insufficient nominal reference points",
            "clusters": []
        }

    # SVD reference plane
    centroid = nominal_top.mean(axis=0)
    centered = nominal_top - centroid

    _, _, vh = np.linalg.svd(
        centered,
        full_matrices=False
    )

    normal = vh[-1]

    if normal[2] < 0:
        normal = -normal

    normal /= np.linalg.norm(normal)

    # Signed deviation from nominal plane
    deviations = (flange_points - centroid) @ normal

    tolerance = config["tolerances"][
        "flange_plane_deviation_mm"
    ]

    failure_mask = np.abs(deviations) > tolerance

    failure_points = flange_points[failure_mask]
    failure_deviations = deviations[failure_mask]

    if len(failure_points) == 0:
        return {
            "status": "NO_CANDIDATES",
            "inspected_points": int(len(flange_points)),
            "candidate_points": 0,
            "clusters": []
        }

    # DBSCAN
    failure_cloud = o3d.geometry.PointCloud()
    failure_cloud.points = o3d.utility.Vector3dVector(
        failure_points
    )

    clustering_cfg = config["defect_clustering"]

    labels = np.asarray(
        failure_cloud.cluster_dbscan(
            eps=clustering_cfg["eps_mm"],
            min_points=clustering_cfg["min_points"],
            print_progress=False
        )
    )

    clusters = []

    for cluster_id in sorted(set(labels)):
        if cluster_id == -1:
            continue

        cluster_mask = labels == cluster_id

        cluster_pts = failure_points[cluster_mask]
        cluster_dev = failure_deviations[cluster_mask]

        center = cluster_pts.mean(axis=0)

        clusters.append({
            "cluster_id": int(cluster_id),
            "feature": "horizontal_flange",
            "num_points": int(len(cluster_pts)),
            "center_xyz_mm": center.tolist(),
            "maximum_absolute_deviation_mm": float(
                np.max(np.abs(cluster_dev))
            ),
            "mean_absolute_deviation_mm": float(
                np.mean(np.abs(cluster_dev))
            ),
            "status": "CANDIDATE_DEFECT"
        })

    return {
        "status": "COMPLETED",
        "inspected_points": int(len(flange_points)),
        "candidate_points": int(len(failure_points)),
        "unclustered_points": int(np.sum(labels == -1)),
        "num_clusters": len(clusters),
        "clusters": clusters
    }
'''

DEFECTS_PATH = PROJECT_ROOT / "src/defects.py"
DEFECTS_PATH.parent.mkdir(parents=True, exist_ok=True)
DEFECTS_PATH.write_text(defects_code)

print("Saved:", DEFECTS_PATH)

Saved: /content/drive/MyDrive/Scan-to-CAD_System/src/defects.py


In [52]:
import importlib
import defects

importlib.reload(defects)

defect_results = defects.cluster_flange_defects(
    nominal_mesh,
    registered_scan,
    config
)

import json

print(json.dumps(defect_results, indent=2))

{
  "status": "COMPLETED",
  "inspected_points": 1190,
  "candidate_points": 647,
  "unclustered_points": 6,
  "num_clusters": 1,
  "clusters": [
    {
      "cluster_id": 0,
      "feature": "horizontal_flange",
      "num_points": 641,
      "center_xyz_mm": [
        6.631121402548637,
        36.23019322553954,
        5.928715020002956
      ],
      "maximum_absolute_deviation_mm": 2.6293339509462723,
      "mean_absolute_deviation_mm": 1.4291957568632356,
      "status": "CANDIDATE_DEFECT"
    }
  ]
}


In [53]:
from defects import cluster_flange_defects

# Detect localized flange defects
defect_results = cluster_flange_defects(
    nominal_mesh,
    registered_scan,
    config
)

In [54]:
import importlib
import run_pipeline

importlib.reload(run_pipeline)

OUTPUT_DIR = PROJECT_ROOT / "outputs/end_to_end_with_defects"

report = run_pipeline.run_pipeline(
    cad_path=CAD_PATH,
    scan_path=SCAN_PATH,
    config_path=CONFIG_PATH,
    output_dir=OUTPUT_DIR
)

print("Inspection:", report["inspection"]["overall_status"])

if "defect_detection" in report:
    print("Defect detection:", report["defect_detection"]["status"])
    print("Clusters:", report["defect_detection"]["num_clusters"])
    print("Candidates:", report["defect_detection"]["candidate_points"])

[Open3D WARNING] Too few correspondences (170) after mutual filter, fall back to original correspondences.
Inspection: FAIL
Defect detection: COMPLETED
Clusters: 1
Candidates: 645


In [ ]:
print("Report keys:")
print(list(report.keys()))

print(
    "Defect detection included:",
    "defect_detection" in report
)

In [ ]:
import importlib
import defects

importlib.reload(defects)

defect_results = defects.cluster_flange_defects(
    nominal_mesh,
    registered_scan,
    config
)

print("Status:", defect_results["status"])
print("Clusters:", len(defect_results["clusters"]))
print("Candidate points:", defect_results.get("candidate_points"))

for cluster in defect_results["clusters"]:
    print(cluster)

In [57]:
from pathlib import Path

BENCHMARK_DIR = PROJECT_ROOT / "benchmarks/noise_experiment"
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

print("Benchmark directory:", BENCHMARK_DIR)

Benchmark directory: /content/drive/MyDrive/Scan-to-CAD_System/benchmarks/noise_experiment


In [ ]:
import numpy as np
import open3d as o3d

rng = np.random.default_rng(42)

base_points = np.asarray(raw_scan.points).copy()

noise_levels = [0.0, 0.05, 0.10, 0.20, 0.40]

benchmark_scans = []

for noise_std in noise_levels:

    noise = rng.normal(
        loc=0.0,
        scale=noise_std,
        size=base_points.shape
    )

    noisy_points = base_points + noise

    cloud = o3d.geometry.PointCloud()
    cloud.points = o3d.utility.Vector3dVector(noisy_points)

    filename = f"scan_noise_{noise_std:.2f}.ply"
    path = BENCHMARK_DIR / filename

    o3d.io.write_point_cloud(str(path), cloud)

    benchmark_scans.append({
        "noise_std_mm": noise_std,
        "path": path
    })

    print(f"Saved: {filename}")

In [ ]:
import json
import pandas as pd
from scipy.spatial.transform import Rotation

with open(GROUND_TRUTH_PATH, "r") as f:
    ground_truth = json.load(f)

T_true = np.array(
    ground_truth["scan_pose"]["scan_to_cad_transformation"]
)

benchmark_results = []

for experiment in benchmark_scans:

    noise_std = experiment["noise_std_mm"]

    print(f"\nTesting additional noise σ = {noise_std:.2f} mm")

    test_scan = o3d.io.read_point_cloud(
        str(experiment["path"])
    )

    aligned_scan, diagnostics = registration.register_scan(
        nominal_mesh,
        test_scan,
        config
    )

    T_est = np.array(diagnostics["transformation"])

    R_delta = T_est[:3, :3] @ T_true[:3, :3].T

    rotation_error = np.degrees(
        Rotation.from_matrix(R_delta).magnitude()
    )

    translation_error = np.linalg.norm(
        T_est[:3, 3] - T_true[:3, 3]
    )

    benchmark_results.append({
        "additional_noise_std_mm": noise_std,
        "rotation_error_deg": rotation_error,
        "translation_error_mm": translation_error,
        "fitness": diagnostics["final_fitness"],
        "rmse_mm": diagnostics["final_rmse_mm"]
    })

    print(f"Rotation error: {rotation_error:.4f}°")
    print(f"Translation error: {translation_error:.4f} mm")

In [60]:
df = pd.DataFrame(benchmark_results)

display(df.round(4))

csv_path = BENCHMARK_DIR / "noise_benchmark.csv"
df.to_csv(csv_path, index=False)

print("Saved:", csv_path)

,additional_noise_std_mm,rotation_error_deg,translation_error_mm,fitness,rmse_mm
0,0.00,0.2499,0.1966,0.9868,0.6874
1,0.05,0.3046,0.1472,0.9868,0.6807
2,0.10,0.3115,0.1415,0.9866,0.6913
3,0.20,0.2188,0.1504,0.9866,0.7077
4,0.40,0.2416,0.2008,0.9856,0.7778


Saved: /content/drive/MyDrive/Scan-to-CAD_System/benchmarks/noise_experiment/noise_benchmark.csv


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(
    df["additional_noise_std_mm"],
    df["translation_error_mm"],
    marker="o"
)

plt.xlabel("Additional Gaussian noise σ (mm)")
plt.ylabel("Translation error (mm)")
plt.title("Registration Robustness vs Additional Scanner Noise")
plt.grid(alpha=0.3)

plt.show()

In [ ]:
import numpy as np
import open3d as o3d
from pathlib import Path

DROPOUT_DIR = PROJECT_ROOT / "benchmarks/dropout_experiment"
DROPOUT_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(123)

base_points = np.asarray(raw_scan.points).copy()

dropout_levels = [0.0, 0.10, 0.25, 0.40, 0.60]

dropout_scans = []

for dropout in dropout_levels:

    keep_probability = 1.0 - dropout

    mask = rng.random(len(base_points)) < keep_probability

    remaining_points = base_points[mask]

    cloud = o3d.geometry.PointCloud()
    cloud.points = o3d.utility.Vector3dVector(remaining_points)

    path = DROPOUT_DIR / f"scan_dropout_{dropout:.2f}.ply"

    o3d.io.write_point_cloud(str(path), cloud)

    dropout_scans.append({
        "dropout_fraction": dropout,
        "path": path,
        "num_points": len(remaining_points)
    })

    print(
        f"Dropout: {dropout*100:.0f}% | "
        f"Remaining points: {len(remaining_points)}"
    )

In [ ]:
import pandas as pd
from scipy.spatial.transform import Rotation

dropout_results = []

for experiment in dropout_scans:

    dropout = experiment["dropout_fraction"]

    print(f"\nTesting dropout = {dropout*100:.0f}%")

    test_scan = o3d.io.read_point_cloud(
        str(experiment["path"])
    )

    try:
        _, diagnostics = registration.register_scan(
            nominal_mesh,
            test_scan,
            config
        )

        T_est = np.array(diagnostics["transformation"])

        R_delta = T_est[:3, :3] @ T_true[:3, :3].T

        rotation_error = np.degrees(
            Rotation.from_matrix(R_delta).magnitude()
        )

        translation_error = np.linalg.norm(
            T_est[:3, 3] - T_true[:3, 3]
        )

        dropout_results.append({
            "dropout_fraction": dropout,
            "remaining_points": experiment["num_points"],
            "rotation_error_deg": rotation_error,
            "translation_error_mm": translation_error,
            "fitness": diagnostics["final_fitness"],
            "rmse_mm": diagnostics["final_rmse_mm"],
            "status": "COMPLETED"
        })

        print(f"Rotation error: {rotation_error:.4f}°")
        print(f"Translation error: {translation_error:.4f} mm")

    except Exception as e:

        dropout_results.append({
            "dropout_fraction": dropout,
            "remaining_points": experiment["num_points"],
            "rotation_error_deg": np.nan,
            "translation_error_mm": np.nan,
            "fitness": np.nan,
            "rmse_mm": np.nan,
            "status": "ERROR"
        })

        print("Registration error:", str(e))

In [ ]:
dropout_df = pd.DataFrame(dropout_results)

display(dropout_df.round(4))

csv_path = DROPOUT_DIR / "dropout_benchmark.csv"

dropout_df.to_csv(csv_path, index=False)

print("Saved:", csv_path)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(
    dropout_df["dropout_fraction"] * 100,
    dropout_df["translation_error_mm"],
    marker="o"
)

plt.xlabel("Random point dropout (%)")
plt.ylabel("Translation error (mm)")
plt.title("Registration Robustness vs Random Point Dropout")

plt.grid(alpha=0.3)
plt.show()

In [ ]:
import numpy as np

def classify_pose(row):
    if row["status"] == "ERROR":
        return "ERROR"

    rot = row["rotation_error_deg"]
    trans = row["translation_error_mm"]

    if not np.isfinite(rot) or not np.isfinite(trans):
        return "ERROR"

    if rot < 2.0 and trans < 2.0:
        return "SUCCESS"

    return "POSE_FAILURE"


dropout_df["benchmark_outcome"] = dropout_df.apply(
    classify_pose,
    axis=1
)

display(dropout_df.round(4))

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation

same_scan_results = []

test_scan = o3d.io.read_point_cloud(
    str(dropout_scans[0]["path"])
)

for trial in range(10):

    print(f"\nTrial {trial + 1}/10")

    _, diag = registration.register_scan(
        nominal_mesh,
        test_scan,
        config
    )

    T_est = np.asarray(diag["transformation"])

    R_delta = T_est[:3, :3] @ T_true[:3, :3].T

    rot_error = np.degrees(
        Rotation.from_matrix(R_delta).magnitude()
    )

    trans_error = np.linalg.norm(
        T_est[:3, 3] - T_true[:3, 3]
    )

    success = (
        rot_error < 2.0 and
        trans_error < 2.0
    )

    same_scan_results.append({
        "trial": trial + 1,
        "rotation_error_deg": rot_error,
        "translation_error_mm": trans_error,
        "fitness": diag["final_fitness"],
        "rmse_mm": diag["final_rmse_mm"],
        "pose_success": success
    })

    print(
        f"Rotation: {rot_error:.3f}° | "
        f"Translation: {trans_error:.3f} mm | "
        f"Success: {success}"
    )

same_scan_df = pd.DataFrame(same_scan_results)

display(same_scan_df.round(4))

print(
    "\nObserved success rate:",
    100 * same_scan_df["pose_success"].mean(),
    "%"
)

In [ ]:
import numpy as np
import registration

good_transform = None
bad_transform = None

for trial in range(20):

    _, diag = registration.register_scan(
        nominal_mesh,
        test_scan,
        config
    )

    T_est = np.array(diag["transformation"])

    rot_error, trans_error = calculate_pose_error(
        T_est,
        T_true
    )

    if rot_error < 2.0 and trans_error < 2.0:
        if good_transform is None:
            good_transform = T_est.copy()
            print("Captured good transformation")

    else:
        if bad_transform is None:
            bad_transform = T_est.copy()
            print("Captured bad transformation")

    if good_transform is not None and bad_transform is not None:
        break

print("Good captured:", good_transform is not None)
print("Bad captured:", bad_transform is not None)

In [69]:
import copy

good_scan = None
bad_scan = None

if good_transform is not None:
    good_scan = copy.deepcopy(test_scan)
    good_scan.transform(good_transform)

if bad_transform is not None:
    bad_scan = copy.deepcopy(test_scan)
    bad_scan.transform(bad_transform)

In [ ]:
import features

for label, scan in [
    ("GOOD REGISTRATION", good_scan),
    ("BAD REGISTRATION", bad_scan)
]:
    if scan is None:
        continue

    points = np.asarray(scan.points)

    print(f"\n{label}")

    for hole_name in ["hole_1", "hole_2"]:

        hole_cfg = config["features"][hole_name]

        result = features.measure_horizontal_hole(
            points,
            approx_y=hole_cfg["approx_center_y_mm"],
            approx_x=hole_cfg["approx_center_x_mm"]
        )

        print(f"\n{hole_name}:")
        print(result)

In [ ]:
from pathlib import Path

PIPELINE_PATH = PROJECT_ROOT / "src/run_pipeline.py"

code = PIPELINE_PATH.read_text()

feature_gate_code = '''

def check_registration_features(registered_scan, config):
    """
    Check whether expected horizontal hole features
    are observable after registration.

    Returns ACCEPTABLE or REVIEW.
    """

    import numpy as np
    from features import measure_horizontal_hole

    points = np.asarray(registered_scan.points)

    diagnostics = {}

    for hole_name in ["hole_1", "hole_2"]:

        hole_cfg = config["features"][hole_name]

        result = measure_horizontal_hole(
            points,
            approx_y=hole_cfg["approx_center_y_mm"],
            approx_x=hole_cfg["approx_center_x_mm"]
        )

        diagnostics[hole_name] = result

        if result["status"] != "MEASURED":
            return "REVIEW", diagnostics

    return "ACCEPTABLE", diagnostics
'''

if "def check_registration_features(" not in code:
    marker = "\ndef run_pipeline("

    assert marker in code, "Could not locate run_pipeline()"

    code = code.replace(
        marker,
        feature_gate_code + marker
    )

PIPELINE_PATH.write_text(code)

print("Feature-aware validation function added!")

In [ ]:
import importlib
import run_pipeline

importlib.reload(run_pipeline)

for name, scan in [
    ("GOOD", good_scan),
    ("BAD", bad_scan)
]:
    status, details = (
        run_pipeline.check_registration_features(
            scan,
            config
        )
    )

    print(f"\n{name} registration")
    print("Feature validation:", status)

    for hole_name, result in details.items():
        print(
            hole_name,
            result["status"],
            result.get("num_points")
        )

In [ ]:
import importlib
import run_pipeline

importlib.reload(run_pipeline)

OUTPUT_DIR = PROJECT_ROOT / "outputs/feature_validated_test"

report = run_pipeline.run_pipeline(
    cad_path=CAD_PATH,
    scan_path=SCAN_PATH,
    config_path=CONFIG_PATH,
    output_dir=OUTPUT_DIR
)

print("Overall inspection:", report["inspection"]["overall_status"])

feature_info = report["registration"].get("feature_validation", {})

print(
    "Feature validation:",
    feature_info.get("status", "NOT RUN")
)

print(
    "Defect detection included:",
    "defect_detection" in report
)

In [ ]:
import numpy as np
import pandas as pd
import registration
import run_pipeline

gate_results = []

for trial in range(10):

    print(f"\nTrial {trial + 1}/10")

    aligned_scan, diag = registration.register_scan(
        nominal_mesh,
        raw_scan,
        config
    )

    # Ground-truth pose error — benchmark only
    T_est = np.asarray(diag["transformation"])

    rot_error, trans_error = calculate_pose_error(
        T_est,
        T_true
    )

    pose_correct = (
        rot_error < 2.0 and
        trans_error < 2.0
    )

    # Operational checks — no ground truth used
    basic_status = run_pipeline.check_registration(
        diag,
        config
    )

    if basic_status == "ACCEPTABLE":
        feature_status, feature_details = (
            run_pipeline.check_registration_features(
                aligned_scan,
                config
            )
        )
    else:
        feature_status = "NOT_RUN"
        feature_details = {}

    final_status = (
        "ACCEPTABLE"
        if basic_status == "ACCEPTABLE"
        and feature_status == "ACCEPTABLE"
        else "REVIEW"
    )

    gate_results.append({
        "trial": trial + 1,
        "rotation_error_deg": rot_error,
        "translation_error_mm": trans_error,
        "pose_correct": bool(pose_correct),
        "fitness": diag["final_fitness"],
        "basic_gate": basic_status,
        "feature_gate": feature_status,
        "final_gate": final_status
    })

    print(
        f"Pose correct: {pose_correct} | "
        f"Basic gate: {basic_status} | "
        f"Feature gate: {feature_status} | "
        f"Final: {final_status}"
    )

In [ ]:
gate_df = pd.DataFrame(gate_results)

false_accept = (
    (~gate_df["pose_correct"]) &
    (gate_df["final_gate"] == "ACCEPTABLE")
)

false_reject = (
    (gate_df["pose_correct"]) &
    (gate_df["final_gate"] != "ACCEPTABLE")
)

correct_accept = (
    (gate_df["pose_correct"]) &
    (gate_df["final_gate"] == "ACCEPTABLE")
)

correct_reject = (
    (~gate_df["pose_correct"]) &
    (gate_df["final_gate"] != "ACCEPTABLE")
)

print("Total trials:", len(gate_df))
print("Correct acceptances:", correct_accept.sum())
print("Correct rejections:", correct_reject.sum())
print("False acceptances:", false_accept.sum())
print("False rejections:", false_reject.sum())

display(gate_df.round(4))

In [ ]:
incorrect_count = (~gate_df["pose_correct"]).sum()
correct_count = gate_df["pose_correct"].sum()

false_accept_rate = (
    false_accept.sum() / incorrect_count
    if incorrect_count > 0 else np.nan
)

false_reject_rate = (
    false_reject.sum() / correct_count
    if correct_count > 0 else np.nan
)

print(
    "False acceptance rate:",
    f"{false_accept_rate:.1%}"
    if np.isfinite(false_accept_rate)
    else "N/A — no incorrect poses"
)

print(
    "False rejection rate:",
    f"{false_reject_rate:.1%}"
    if np.isfinite(false_reject_rate)
    else "N/A — no correct poses"
)

In [ ]:
from pathlib import Path

GATE_BENCHMARK_DIR = (
    PROJECT_ROOT / "benchmarks/registration_gate"
)

GATE_BENCHMARK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

csv_path = GATE_BENCHMARK_DIR / "gate_benchmark.csv"

gate_df.to_csv(csv_path, index=False)

print("Saved:", csv_path)

In [ ]:
import numpy as np
import open3d as o3d

good_points = np.asarray(good_scan.points)

# Approximate Hole 1 center in CAD coordinates
hole_y = 15.5
hole_x = 6.5

radial_distance = np.sqrt(
    (good_points[:, 1] - hole_y)**2 +
    (good_points[:, 0] - hole_x)**2
)

# Remove points from Hole 1's cylindrical wall region
occlusion_mask = (
    (radial_distance < 3.5) &
    (good_points[:, 2] > 1.0) &
    (good_points[:, 2] < 3.5)
)

occluded_points = good_points[~occlusion_mask]

occluded_scan = o3d.geometry.PointCloud()
occluded_scan.points = o3d.utility.Vector3dVector(
    occluded_points
)

print("Original points:", len(good_points))
print("Occluded points:", len(occluded_points))
print("Removed points:", np.sum(occlusion_mask))

In [ ]:
status, details = (
    run_pipeline.check_registration_features(
        occluded_scan,
        config
    )
)

print("Feature validation:", status)

for hole_name, result in details.items():
    print(
        hole_name,
        result["status"],
        result.get("num_points")
    )

In [80]:
for name, scan in [
    ("Original correct registration", good_scan),
    ("Correct registration with occlusion", occluded_scan)
]:
    status, details = (
        run_pipeline.check_registration_features(
            scan,
            config
        )
    )

    print(f"\n{name}")
    print("Status:", status)

    for hole_name, result in details.items():
        print(
            hole_name,
            "Points:", result.get("num_points"),
            "Status:", result["status"]
        )


Original correct registration
Status: ACCEPTABLE
hole_1 Points: 81 Status: MEASURED
hole_2 Points: 89 Status: MEASURED

Correct registration with occlusion
Status: REVIEW
hole_1 Points: 0 Status: REVIEW
